# Few-shot Discord event announcement (`ai_query`)

**Purpose:** Build a Japanese Discord-style event announcement using the few-shot prompt layout from `docs/few_shot_discord_event_announcement_samples.md`, then call Databricks **`ai_query`** on a Foundation Model endpoint.

**Author:** Cheng Wang  
**Contact:** cheng.wang@myteam.com  
**Date:** 2026-03-23  

## Prerequisites

- Workspace in a region where [AI Functions / `ai_query`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query) are available.
- Your user or job identity has **CAN QUERY** on the configured foundation endpoint.
- Cluster / notebook uses a **Databricks Runtime** that supports `ai_query` (and `modelParameters` if used; DBR 15.3+).
- Adjust **`FOUNDATION_ENDPOINT`**, **`TEMPERATURE`**, and **`MAX_TOKENS`** in the constants cell if your workspace uses different defaults.

## Prompt source

Static few-shot blocks are embedded below (self-contained on the cluster).  
`Source: docs/few_shot_discord_event_announcement_samples.md`


## Web app contract (later)

| Notebook widget   | Client (form / JSON)     |
|-------------------|--------------------------|
| `tone`            | Same six hyperparameters |
| `length`          | as dropdown fields       |
| `formality`       |                          |
| `emoji_density`   |                          |
| `structure`       |                          |
| `cta_strength`    |                          |
| `user_request`    | Natural-language field   |

**Server-side only (not from client):** context facts (`CONTEXT_FOR_REQUEST`), model endpoint name, `TEMPERATURE`, `MAX_TOKENS` — same as the constants in this notebook. Build the full prompt server-side, then run one `ai_query` row (or equivalent SQL / Jobs API).


In [0]:
# --- Fixed in code (edit per environment / event) ---
# Source: docs/few_shot_discord_event_announcement_samples.md

FOUNDATION_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.3
MAX_TOKENS = 2048


In [0]:
# Widgets: six behavior hyperparameters + natural language request
dbutils.widgets.dropdown("tone", "カジュアル", ["真面目", "おふざけ", "カジュアル"], "Tone")
dbutils.widgets.dropdown("length", "medium", ["short", "medium", "long"], "Length")
dbutils.widgets.dropdown("formality", "ですます", ["ですます", "タメ口", "敬語"], "Formality")
dbutils.widgets.dropdown("emoji_density", "普通", ["なし", "少なめ", "普通", "多め"], "Emoji density")
dbutils.widgets.dropdown("structure", "箇条書き中心", ["箇条書き中心", "段落中心", "見出し＋本文"], "Structure")
dbutils.widgets.dropdown("cta_strength", "普通", ["控えめ", "普通", "強め"], "CTA strength")
dbutils.widgets.text(
    "user_request",
    "来週土曜21時の練習会告知を、カジュアルで中くらいの長さ、箇条書き中心で作って",
    "User request",
)

tone = dbutils.widgets.get("tone")
length = dbutils.widgets.get("length")
formality = dbutils.widgets.get("formality")
emoji_density = dbutils.widgets.get("emoji_density")
structure = dbutils.widgets.get("structure")
cta_strength = dbutils.widgets.get("cta_strength")
user_request = dbutils.widgets.get("user_request")


In [0]:
# Static few-shot core (System + single-param + combined + output rules)
# Source: docs/few_shot_discord_event_announcement_samples.md

STATIC_PROMPT_CORE = 'STATIC_PROMPT_CORE = '[System / Role]\nYou are a post writer for a game community. Follow predefined behavior settings and examples.\n\n[Hyperparameter Definitions]\n- Tone: 真面目 / おふざけ / カジュアル\n- Length: short / medium / long\n  - short: 〜100字\n  - medium: 101〜300字\n  - long: 301字〜\n- Formality: ですます / タメ口 / 敬語\n- Emoji density: なし / 少なめ / 普通 / 多め\n  - なし=0個, 少なめ=1-2個, 普通=3-5個, 多め=6個以上\n- Structure: 箇条書き中心 / 段落中心 / 見出し＋本文\n- Call-to-action strength: 控えめ / 普通 / 強め\n\n[Single-parameter Examples (few-shot; 全項目)]\n- Tone examples:\n  --- 真面目 例1（原文）---\n  # 🎮 **早慶戦開催！！！！！！** 🎮\n\n  皆さんこんにちは！\n  春休みいかがお過ごしでしょうか。\n\n  春休みももう既に後半。\n  そんな春休みの最後に最高の思い出として\n  **2025年度 eスポーツ早慶戦** を開催します！！🔥\n\n  ## 📅 **イベント詳細**\n\n  **📍 日にち**：3/13(金) 12:00～\n  **📍 場所**：esports Style UENO\n  **📍 種目**：**VALORANT**\n\n  ## 🎁 **今回の見どころ**\n\n  実はまだ秘密でしたが…今回はなんと！\n\n  * **東プレ様のキーボード** 🎮\n  * **VCTキャスターの yueさん** 🎤\n  * **人気ストリーマーの xnfriさん** 🔥\n\n  が来場予定です！！\n\n  ## 🙌 **ぜひ来てください！**\n\n  ### ▼来場登録はこちら\n\n  https://forms.gle/QLQktapD9MM1JgkX7\n\n  ## ⚠️ **注意事項**\n\n  * ⏰ **回答締切：3/12 23:59まで**\n  * 🚶\u200d♂️ **途中参加・退出OK**\n  * 🏃 **当日飛び込み参加も可能（状況次第）**\n\n  みなさんのご参加、お待ちしています！！🔥🔥🔥\n\n  --- 真面目 例2（原文）---\n  # 🔥 **早慶戦 当日スタッフ追加募集** 🔥\n\n  ## 💪 **あなたのご協力が力になります！**\n\n  **早慶戦の当日スタッフを募集します！** 🔥\n\n  ### 🎮 **主なお仕事**\n\n  * **ゲームオブザーバー**（試合中の選手視点）\n  * **カメラ**（ステージ上の選手や会場の様子を配信に反映）\n\n  などを担当していただきます！\n\n  ## 🌱 **未経験でもOK！**\n\n  * 経験は不要です\n  * 当日の動き・台本などは丁寧に説明します\n\n  eスポーツの裏側や本格的な現場に触れられる\n  **貴重なオフライン体験**になります！\n\n  ## 📩 **応募方法**\n\n  少しでも興味がある方は\n  👉 **@Kar_10 までご連絡ください！** 🙌\n\n  ご協力、ぜひよろしくお願いします！🔥\n\n  --- 真面目 例3（原文）---\n  # 🧩 **早慶戦 制作メンバー募集** 🧩\n\n  お疲れ様です！\n  公式からの告知にもあった通り、今年度も早慶戦の開催が決定しました！\n  それに伴い、**制作メンバーの募集**を行います！\n\n  ## 📌 **制作メンバーって？**\n\n  eスポーツ早慶戦という**オフラインイベントを一緒に作るメンバー**です！\n  主に当日スタッフを広く募集しますが、\n  関心や余力に合わせて様々な役割を担当できます！\n\n  ## 🎨 **例えばこんな仕事があります**\n\n  * カメラマン（配信・記録）\n  * オブザーバー（VALORANT）\n  * 映像・音響・照明\n\n  ## 💡 **こんな方におすすめ！**\n\n  * イベントの企画・運営が好き\n  * 映像・音響・照明機材に触れた経験がある\n  * OBSなどで配信したことがある\n  * 動画制作の経験がある\n\n  TTZからのテクニカル人材も在籍しているため、\n  未経験でも安心して参加できます！\n\n  ## 📩 **応募方法**\n\n  ぜひ応募フォームをご確認ください！\n\n  👉 https://forms.gle/FEbftMZbjDspqFwF6\n\n  ご参加お待ちしています！🔥\n\n  --- おふざけ 例1（原文）---\n  # 🎮 **6/29(日) VALOダイヤ以下大会「カメムシ未満最強決定戦」開催！** 🎮\n\n  @FPS部門 @VALORANT\n\n  上位ランクじゃなくても部内大会で思いっきり活躍したい！\n  そんな声を受けて、**今年度初のランク制限あり大会**を開催します！💥\n\n  🐢 **カメムシ以上の方は観戦でお願いします！** 🐢\n\n  ## 📅 **イベント詳細**\n\n  * **日時**：6/29(日) 21:00集合\n  * **参加条件**：過去最高ランクが**ダイヤ以下（ダイヤ3も可）**\n  * **賞品**：優勝チームに\n    👉 **アマギフ 1000円 ×5名分**\n    👉 **TitanZz公式Twitterからのフォロバ**\n\n  ## 🙌 **参加方法**\n\n  **個人エントリー制**です！\n  参加したい方はこの投稿に**リアクションスタンプ 👍**をお願いします！\n\n  みなさんの参加お待ちしています！🔥\n\n  --- おふざけ 例2（原文）---\n  @FPS部門 @LOL部門 @格ゲー部門 @仮部員 \n\n  # :rotating_light:TitanZz緊急ミッション発令:rotating_light: #\n  ## **:milky_way:Among Us宇宙人狼大会 @GW:milky_way:** ##\n  ～裏切り者は、誰だッ！？～\n\n  **:billed_cap:JuJu艦長からの緊急通達:billed_cap:**\n  「……船内で異常反応。念のため、緊急会議を開く。」\n\n  乗組員たちがくつろいでいたそのとき、\n  JuJu艦長が静かに、“あのボタン”に手をかけた――\n\n  **:red_circle:緊急会議 発動:red_circle:**\n  「裏切り者は、この中にいる。」\n\n  **:flying_saucer:TitanZz宇宙船、5月5日 21:00 出航予定**\n  信じ合う（フリをする）仲間たちと、疑心暗鬼の旅へ。\n\n  :video_game: ゲーム：Among Us（スマホ・PCどちらでもOK）\n  :calendar_spiral: 日程：5月5日（月）21:00～\n  :round_pushpin: 場所：TitanZz Discordサーバー\n\n  :knife:**誰がインポスターなのか、誰がガチで頭いいのか、そして誰がただのバカなのか**――\n  全てが暴かれる夜が始まる。\n\n  :flying_saucer:参加方法：この投稿にリアクションでOK！\n\n  :star2: 初心者・初見 大歓迎！\n  :star2: エンジョイ勢の参加も余裕でアリ！\n  :star2: もちろん途中参加・途中離脱OK！\n\n  --- おふざけ 例3（原文）---\n  @仮部員 @FPS部門 \n  # :cyclone: FPS部門 緊急会議:loudspeaker: #\n  ## 今週、俺たちは何を撃つべきか——！？ ##\n\n  VALO…OW2…APEX…？それともまさかのCS2？？\n  今週のTitanZz FPSイベント、やるゲームは\n  :video_game: ＼みんなの投票で決まります！／\n\n  「やるゲームが変わる、それは君のリアクション次第。」\n  リアクションで清き一票を！（1人1票でお願いします）:boom:\n  **※新入部員のリアクションは1.5票分の価値あり！（ようこそ特権）\n  **\n  :date: **開催日：4月19日（土）21:00〜**\n  :ballot_box: ▼下のリアクションから投票してね！（締切：当日18:00まで）\n  :blue_circle:：VALORANT\n  :orange_circle:：Overwatch 2\n  :purple_circle:：APEX\n  :green_circle:：CS2\n  （その他希望タイトルがあればリプライでどうぞ！）\n\n  --- カジュアル 例1（原文）---\n  # 🎮 **春新歓目前！VALORANT交流カスタム開催決定！** 🎮\n\n  「春の新歓に向けて、まずは内部で盛り上がっていこう！」\n  ということで、**VALOカスタムを開催します！** 🔥\n\n  日程調整のため、下の候補日程に\n  **参加可能な日にリアクションしてください！**\n\n  ## 🏆 **開催概要**\n\n  * **日程**：リアクションで調整（20:00〜23:00頃予定）\n  * **賞品**：優勝チームに**アマギフ1000円分プレゼント！**\n  * **特典**：スカーミッシュ最強ロール付与🔥\n\n  ## ⚔️ **特殊対戦ルール（3人1組チーム戦）**\n\n  今回は通常のスカーミッシュではなく、\n  **3人1組のチーム対抗 BO3形式**で実施します！\n\n  1. **第1ゲーム**：リーダー同士のガチンコ **1v1**\n  2. **第2ゲーム**：リーダー以外の2人による連携 **2v2**\n  3. **第3ゲーム**：チーム全員での総力戦 **3v3**\n\n  初心者の方でも活躍できるよう、\n  運営側でしっかり調整します！💪\n\n  ## 📝 **日程調整について**\n\n  この投稿へのリアクション＝参加確定ではありません！\n  参加できる日に**気軽にリアクション**してください！🙌\n\n  皆さんの参加、お待ちしています！🔥\n\n  --- カジュアル 例2（原文）---\n  # 📢 **「第2回 早慶交流会🐻×🦅」開催決定！** 🔥\n\n  早稲田大学esportsサークルWEC🐻の皆さんとの\n  **第2回交流会の開催が決定しました！！**\n\n  「第1回からまだ仲良くなりきれてない…」\n  そんな方も多いのではないでしょうか？👀\n\n  今回は**ランク制限を撤廃！**\n  より多くの人と関われるようにします！\n\n  みんなで早稲田の方たちと仲良くなって、\n  もっと楽しいゲームライフを送りましょう！！🎮\n\n  さらに今回は、\n  **早慶戦メンバーによるカスタム**も実施予定です！🔥\n\n  ## 📅 **イベント詳細**\n\n  * **日程**：11月1日(土) 21:00〜（途中参加OK）\n  * **場所**：TitanZz外部交流サーバー（オンライン）\n  * **ゲーム**：**VALORANT**\n\n  ## 📝 **形式**\n\n  今回は**ランク制限なし！**\n  誰でも気軽に参加できます！\n\n  ## 🙌 **参加方法**\n\n  ▼参加希望の方はこちらのフォームから\n  https://docs.google.com/forms/d/e/1FAIpQLSebP3hwY8h-9NJVbAAXdusFycE9gded0bhVZRzDDvJyDwn3g/viewform?usp=header\n\n  ▼外部交流サーバーはこちら\n  https://discord.gg/sYdjs3DzV\n\n  @FPS部門 @VALORANT\n\n  皆さんの参加お待ちしています！🔥\n\n  --- カジュアル 例3（原文）---\n  # 🎉 **TitanZz 対面新歓ボランティア再募集！！** 🎶✨\n\n  新歓は、**TitanZzを盛り上げる大チャンス！**\n  新入生にサークルの魅力を伝えて、\n  一緒に活動する仲間を増やしましょう！🙌\n\n  参加してくれる人が増えれば増えるほど、\n  新入生もたくさん来てくれるかも💪\n\n  ## 🎯 **当日の役割**\n\n  * 新入生からの質問対応（経験を話すだけでもOK！）\n  * ビラ配り＆雰囲気作り（楽しく話しかけるだけでOK！）\n\n  ## 📅 **日程（参加できるところにリアクション！）**\n\n  ### 🟠 日吉キャンパス\n\n  * 4/2(木) 15:45〜17:15\n  * 4/3(金) 12:15〜13:45\n  * 4/6(月) 15:45〜17:15\n\n  ### 🟢 SFCキャンパス\n\n  * 4/7(火) 13:30〜16:30\n\n  ## ⏰ **締め切り**\n\n  * **3月31日 23:59**\n\n  ## 🙌 **参加方法**\n\n  日程のリアクションスタンプを押して参加！\n\n  できるだけ多くの部員のご協力をお待ちしています🔥\n  ぜひお手伝いよろしくお願いします！🥺\n\n- Formality examples:\n  - ですます: 「本日20時から練習会を開催します。ご参加お待ちしています。」\n  - タメ口: 「今日20時から練習会やるよ。来れたら来て！」\n  - 敬語: 「本日20時より練習会を開催いたします。ご参加のほどよろしくお願いいたします。」\n- Structure examples（形式別・原文・各パターン3例）:\n  --- 箇条書き中心 例1（原文）---\n  # 🎉 **TitanZz 対面新歓ボランティア再募集！！** 🎶✨\n\n  ## 🎯 **当日の役割**\n\n  * 新入生からの質問対応（経験を話すだけでもOK！）\n  * ビラ配り＆雰囲気作り（楽しく話しかけるだけでOK！）\n\n  ## 📅 **日程（参加できるところにリアクション！）**\n\n  ### 🟠 日吉キャンパス\n\n  * 4/2(木) 15:45〜17:15\n  * 4/3(金) 12:15〜13:45\n  * 4/6(月) 15:45〜17:15\n\n  ### 🟢 SFCキャンパス\n\n  * 4/7(火) 13:30〜16:30\n\n  ## ⏰ **締め切り**\n\n  * **3月31日 23:59**\n\n  ## 🙌 **参加方法**\n\n  --- 箇条書き中心 例2（原文）---\n  # 🎮 **春新歓目前！VALORANT交流カスタム開催決定！** 🎮\n\n  ## 🏆 **開催概要**\n\n  * **日程**：リアクションで調整（20:00〜23:00頃予定）\n  * **賞品**：優勝チームに**アマギフ1000円分プレゼント！**\n  * **特典**：スカーミッシュ最強ロール付与🔥\n\n  ## ⚔️ **特殊対戦ルール（3人1組チーム戦）**\n\n  今回は通常のスカーミッシュではなく、\n  **3人1組のチーム対抗 BO3形式**で実施します！\n\n  1. **第1ゲーム**：リーダー同士のガチンコ **1v1**\n  2. **第2ゲーム**：リーダー以外の2人による連携 **2v2**\n  3. **第3ゲーム**：チーム全員での総力戦 **3v3**\n\n  ## 📝 **日程調整について**\n  この投稿へのリアクション＝参加確定ではありません！\n  参加できる日に**気軽にリアクション**してください！🙌\n\n  --- 箇条書き中心 例3（原文）---\n  # 🎮 **6/29(日) VALOダイヤ以下大会「カメムシ未満最強決定戦」開催！** 🎮\n\n  @FPS部門 @VALORANT\n  ## 📅 **イベント詳細**\n\n  * **日時**：6/29(日) 21:00集合\n  * **参加条件**：過去最高ランクが**ダイヤ以下（ダイヤ3も可）**\n  * **賞品**：優勝チームに\n    👉 **アマギフ 1000円 ×5名分**\n    👉 **TitanZz公式Twitterからのフォロバ**\n\n  ## 🙌 **参加方法**\n\n  **個人エントリー制**です！\n\n  --- 段落中心 例1（原文）---\n  【いよいよ明日📢】\n  esports Style UENOでeスポーツ早慶戦が開催！🔫\n\n  早稲田大学esportsサークル⚔️慶應義塾大学TitanZz\n  @esportswsd\n   vs \n  @TitanZz_Esports\n\n\n  #eスポーツ早慶戦 #VALORANT\n\n  --- 段落中心 例2（原文）---\n  WE WANT YOU!!\n  eスポーツサークルTitanZzでは一緒にプレイする慶應生を募集しています！🔥\n\n  新入生も上級生も大歓迎🌸初心者、ガチ勢、エンジョイ勢どなたでもどうぞ！\n  少しでも興味のある方はDMより気軽にご連絡ください！\n\n  #慶應 #春から慶應 #春から慶応 #sfc26 #慶應新歓\n\n  --- 段落中心 例3（原文）---\n  ◤お知らせ◢\n\n  eスポーツ早慶戦\n  公式Instagramを開設しました。\n\n  そして——\n  Instagramにてキービジュアルも初公開。\n\n  神話か。伝説か。\n  その戦いが、ここから始まる。\n\n  ▼Instagram\n\n  https://instagram.com/sokei_esports?igsh=Y3hyMGd4aWtuc3Vv&utm_source=qr\n\n  #eスポーツ早慶戦  #VALORANT \n  #慶應  #早稲田\n\n  --- 見出し＋本文 例1（原文）---\n  # 🧩 **早慶戦 制作メンバー募集** 🧩\n\n  お疲れ様です！\n  公式からの告知にもあった通り、今年度も早慶戦の開催が決定しました！\n  それに伴い、**制作メンバーの募集**を行います！\n\n  ## 📌 **制作メンバーって？**\n\n  eスポーツ早慶戦という**オフラインイベントを一緒に作るメンバー**です！\n  主に当日スタッフを広く募集しますが、\n  関心や余力に合わせて様々な役割を担当できます！\n\n  ## 🎨 **例えばこんな仕事があります**\n\n  * カメラマン（配信・記録）\n  * オブザーバー（VALORANT）\n  * 映像・音響・照明\n\n  ## 💡 **こんな方におすすめ！**\n\n  * イベントの企画・運営が好き\n  * 映像・音響・照明機材に触れた経験がある\n  * OBSなどで配信したことがある\n  * 動画制作の経験がある\n\n  TTZからのテクニカル人材も在籍しているため、\n  未経験でも安心して参加できます！\n\n  ## 📩 **応募方法**\n\n  ぜひ応募フォームをご確認ください！\n\n  👉 https://forms.gle/FEbftMZbjDspqFwF6\n\n  ご参加お待ちしています！🔥\n\n  --- 見出し＋本文 例2（原文）---\n  # 🎮 **早慶戦開催！！！！！！** 🎮\n\n  皆さんこんにちは！\n  春休みいかがお過ごしでしょうか。\n\n  春休みももう既に後半。\n  そんな春休みの最後に最高の思い出として\n  **2025年度 eスポーツ早慶戦** を開催します！！🔥\n\n  ## 📅 **イベント詳細**\n\n  **📍 日にち**：3/13(金) 12:00～\n  **📍 場所**：esports Style UENO\n  **📍 種目**：**VALORANT**\n\n  ## 🎁 **今回の見どころ**\n\n  実はまだ秘密でしたが…今回はなんと！\n\n  * **東プレ様のキーボード** 🎮\n  * **VCTキャスターの yueさん** 🎤\n  * **人気ストリーマーの xnfriさん** 🔥\n\n  が来場予定です！！\n\n  ## 🙌 **ぜひ来てください！**\n\n  ### ▼来場登録はこちら\n\n  https://forms.gle/QLQktapD9MM1JgkX7\n\n  ## ⚠️ **注意事項**\n\n  * ⏰ **回答締切：3/12 23:59まで**\n  * 🚶\u200d♂️ **途中参加・退出OK**\n  * 🏃 **当日飛び込み参加も可能（状況次第）**\n\n  みなさんのご参加、お待ちしています！！🔥🔥🔥\n\n  --- 見出し＋本文 例3（原文）---\n  # 🎮 **春新歓目前！VALORANT交流カスタム開催決定！** 🎮\n\n  「春の新歓に向けて、まずは内部で盛り上がっていこう！」\n  ということで、**VALOカスタムを開催します！** 🔥\n\n  日程調整のため、下の候補日程に\n  **参加可能な日にリアクションしてください！**\n\n  ## 🏆 **開催概要**\n\n  * **日程**：リアクションで調整（20:00〜23:00頃予定）\n  * **賞品**：優勝チームに**アマギフ1000円分プレゼント！**\n  * **特典**：スカーミッシュ最強ロール付与🔥\n\n  ## ⚔️ **特殊対戦ルール（3人1組チーム戦）**\n\n  今回は通常のスカーミッシュではなく、\n  **3人1組のチーム対抗 BO3形式**で実施します！\n\n  1. **第1ゲーム**：リーダー同士のガチンコ **1v1**\n  2. **第2ゲーム**：リーダー以外の2人による連携 **2v2**\n  3. **第3ゲーム**：チーム全員での総力戦 **3v3**\n\n  初心者の方でも活躍できるよう、\n  運営側でしっかり調整します！💪\n\n  ## 📝 **日程調整について**\n\n  この投稿へのリアクション＝参加確定ではありません！\n  参加できる日に**気軽にリアクション**してください！🙌\n\n  皆さんの参加、お待ちしています！🔥\n\n- Call-to-action strength examples（末尾1文のみ変更）:\n  - 控えめ: 「参加できる方は、可能であればリアクションをお願いします。」\n  - 普通: 「参加できる方はリアクションをお願いします。」\n  - 強め: 「参加する方は今すぐリアクションしてください！」\n\n[Combined Examples（複合指定→出力結果）]\n- Case 1:\n  - Hyperparameters: Tone=カジュアル, Length=medium, Formality=ですます, Emoji density=普通, Structure=箇条書き中心, CTA=普通\n  - Context: Geoguessr, 5/24 21:00, TitanZz Discord, 参加方法はリアクション\n  - User request: 「来週土曜21時の練習会告知を、カジュアルで中くらいの長さ、箇条書き中心で作って」\n  - Output（原文）:\n    ```text\n    📢 TitanZz Geoguessr 開催決定！  🌍✨,\n    ～配信で話題の“地図当てゲーム”、みんなでワイワイやろう！～,\n     🗓️日程 ：5月24日（土）21:00～\n     🎮ゲーム ：Geoguessr（無料でプレイ！）\n     🚩場所 ：TitanZz Discordサーバー\n    \n    🔹「配信で見たけどやったことない！」って人にも体験してほしい🌱\n    🔹いまハマってる人たちも、もっと盛り上がれたら最高！🚀\n    🔹途中参加・途中離脱・観戦だけでもOK！\n    \n    👉【参加方法】\n    この投稿にリアクションするだけ🙌\n    \n    「ちょっとやってみたいかも…」って人も、「俺の地理力、見せつけるぜ！」って人も、\n    みんなでワイワイやりましょー！初参加＆エンジョイ勢も大歓迎！\n    お待ちしてます🌏🗺️✨\n    ```\n- Case 2:\n  - Hyperparameters: Tone=おふざけ, Length=long, Formality=タメ口, Emoji density=多め, Structure=見出し＋本文, CTA=強め\n  - Context: Among Us, 5/5 21:00, Discord開催\n  - User request: 「宇宙人狼大会の告知を、ノリ強めでお祭り感ある文章にして」\n  - Output（原文）:\n    ```text\n    🚨TitanZz緊急ミッション発令🚨\n    🌌Among Us宇宙人狼大会 @GW🌌\n    ～裏切り者は、誰だッ！？～\n    \n    🧢JuJu艦長からの緊急通達🧢\n    「……船内で異常反応。念のため、緊急会議を開く。」\n    \n    乗組員たちがくつろいでいたそのとき、\n    JuJu艦長が静かに、“あのボタン”に手をかけた――\n    \n    🔴緊急会議 発動🔴\n    「裏切り者は、この中にいる。」\n    \n    🛸TitanZz宇宙船、5月5日 21:00 出航予定\n    信じ合う（フリをする）仲間たちと、疑心暗鬼の旅へ。\n    \n    🎮 ゲーム：Among Us（スマホ・PCどちらでもOK）\n    🗓️ 日程：5月5日（月）21:00～\n    📍 場所：TitanZz Discordサーバー\n    \n    🔪誰がインポスターなのか、誰がガチで頭いいのか、そして誰がただのバカなのか――\n    全てが暴かれる夜が始まる。\n    \n    🛸参加方法：この投稿にリアクションでOK！\n    \n    🌟 初心者・初見 大歓迎！\n    🌟 エンジョイ勢の参加も余裕でアリ！\n    🌟 もちろん途中参加・途中離脱OK！\n    ```\n- Case 3:\n  - Hyperparameters: Tone=真面目, Length=long, Formality=敬語, Emoji density=少なめ, Structure=見出し＋本文, CTA=控えめ\n  - Context: オフラインイベント最終連絡、会場住所あり、集合11:00\n  - User request: 「参加者向け最終連絡を、丁寧で抜け漏れなく書いて」\n  - Output（原文）:\n    ```text\n    📢 @eスポカフェ25春 参加者の皆さまへ\n    明日のオフラインイベントに関する最終連絡です！\n    \n    🎮 タイムスケジュール\n    当日のコンテンツは盛りだくさん！\n    \n    🕚 11:00 ～ 12:30\n    ・受付・フリープレイ\n    \n    🕧 12:30 ～ 14:30\n    ・VALORANT トーナメント（チーム分けは当日アナウンス予定）\n    \n    🕒 15:00 ～ 16:00\n    ・LoL「ムンドドッジボール」\n    ・Street Fighter 6 対戦会（同時進行）\n    ・Overwatch カスタムマッチ（同時進行）\n    \n    ✨すべて観戦もOK！出る人も見る人も楽しめる構成です！\n    \n    📍 場所\n    e-sportscafe LIG\n    〒210-0006 神奈川県川崎市川崎区砂子２−６−１１\n    🗺️ 公式サイト\n    🎥 行き方動画\n    \n    👥 集合時間・連絡先\n    🕚 11:00〜12:00 現地集合！\n    到着したら @Xenon か @JuJu  か @Aru か @Axela に直接 or Discordで連絡ください！\n    \n    ⚠️ 注意事項\n    ・昼食は各自で自由に（持ち込みOK！）\n    ・デバイス持ち込み可（お店の備え付けも使えます）\n    ・途中参加／途中退室OK（事前に一言いただけると助かります！）\n    \n    明日は思いっきり楽しみましょう！！\n    ご不明点があればこの投稿にリプライください\U0001fae1🔥 \n    ```\n- Case 4:\n  - Hyperparameters: Tone=カジュアル, Length=short, Formality=ですます, Emoji density=少なめ, Structure=段落中心, CTA=強め\n  - Context: 8/30-31 OECU CUP 出場告知\n  - User request: 「出場告知を短く、応援お願いを強めで」\n  - Output:\n    「8/30-31にTitanZzがOECU CUPへ出場します！🔥\n    応援してくださる方は、ぜひ拡散とリアクションをお願いします！」 \n- Case 5:\n  - Hyperparameters: Tone=カジュアル, Length=medium, Formality=タメ口, Emoji density=普通, Structure=段落中心, CTA=普通\n  - Context: OECU CUP 優勝報告\n  - User request: 「結果報告を明るく、感謝を入れて書いて」\n  - Output:\n    「🏆 OECU CUP 優勝！\n    2日間の試合を勝ち抜いて優勝できました。応援してくれたみんな本当にありがとう！🙌\n    次も頑張るので引き続きよろしく！」\n''


Re-run cells 7-8 when you change the hyperparameters.

In [0]:
def build_hyperparameter_block(
    tone: str,
    length: str,
    formality: str,
    emoji_density: str,
    structure: str,
    cta_strength: str,
) -> str:
    length_help = (
        "short: 〜100字（1〜2文） / medium: 101〜300字（2〜5文） / long: 301字〜（補足・背景あり）"
    )
    emoji_help = "なし=0個 / 少なめ=1〜2個 / 普通=3〜5個 / 多め=6個以上"
    return f"""[Hyperparameter Definitions — この生成の指定値]
- Tone: {tone}
- Length: {length} ({length_help})
- Formality: {formality}
- Emoji density: {emoji_density} ({emoji_help})
- Structure: {structure}
- Call-to-action strength: {cta_strength}
"""


hyper_block = build_hyperparameter_block(
    tone, length, formality, emoji_density, structure, cta_strength
)

_split = STATIC_PROMPT_CORE.split("[Single-parameter Examples", 1)
_head = _split[0].rstrip()
_single_combined = "[Single-parameter Examples" + _split[1].rsplit("[Output instruction]", 1)[0].rstrip()
_output_instruction = (
    "[Output instruction]" + STATIC_PROMPT_CORE.rsplit("[Output instruction]", 1)[1].strip()
)

full_prompt = (
    _head
    + "\n\n"
    + hyper_block
    + "\n\n"
    + _single_combined
    + "\n\n[User request (natural language)]\n"
    + user_request.strip()
    + "\n\n"
    + _output_instruction
)

In [0]:
df = spark.createDataFrame([(full_prompt,)], ["full_prompt"])

# modelParameters and failOnError require DBR 15.3+; simplify if your runtime errors.
result = df.selectExpr(
    f"ai_query('{FOUNDATION_ENDPOINT}', full_prompt, "
    f"modelParameters => named_struct('temperature', CAST({TEMPERATURE} AS DOUBLE), "
    f"'max_tokens', CAST({MAX_TOKENS} AS INT)), "
    "failOnError => false) AS announcement"
)

row = result.collect()[0]
text = row["announcement"]["result"]
error = row["announcement"]["errorMessage"]

if error:
    print(f"⚠️ ai_query error: {error}")
else:
    # Render with preserved line breaks
    html = text.replace("\n", "<br>")
    displayHTML(f'<div style="white-space:pre-wrap; font-size:14px; line-height:1.6;">{html}</div>')